In [ ]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [ ]:
df_green = spark.read.parquet('data/pq/green/*/*')

```sql
SELECT
    DATE_TRUNC('hour', lpep_pickup_datetime) as hour,
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM green
WHERE lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY 1, 2
```

In [ ]:
rdd = df_green \
    .select('lpep_pickup_datetime', 'PULocationID', 'total_amount') \
    .rdd

In [ ]:
from datetime import datetime

In [ ]:
start = datetime(year=2020, month=1, day=1)
end = datetime(year=2020, month=1, day=10)

def filter_outliers(row):
    return row.lpep_pickup_datetime >= start and row.lpep_pickup_datetime <= end

In [ ]:
def prepare_for_grouping(row):
    hour = row.lpep_pickup_datetime.replace(minute=0, second=0, microsecond=0)
    zone = row.PULocationID
    key = (hour, zone)

    amount = row.total_amount
    count = 1
    value = (amount, count)

    return (key, value)

In [ ]:
def calc_revenue(left, right):
    left_amt, left_count = left
    right_amt, right_count = right

    out_amt = left_amt + right_amt
    out_count = left_count + right_count

    return (out_amt, out_count)

In [ ]:
from collections import namedtuple

In [ ]:
RevenueRow = namedtuple('RevenueRow', ['hour', 'zone', 'revenue', 'count'])

In [ ]:
def unwrap(row):
    # first tuple: [date, zone]; second tuple: [total_rev, count]
    # return (row[0][0], row[0][1], row[1][0], row[1][1])
    return RevenueRow(
        hour=row[0][0],
        zone=row[0][1],
        revenue=row[1][0],
        count=row[1][1]
    )

In [ ]:
from pyspark.sql import types

In [ ]:
result_schema = types.StructType([
    types.StructField('hour', types.TimestampType(), True),
    types.StructField('zone', types.IntegerType(), True),
    types.StructField('revenue', types.DoubleType(), True),
    types.StructField('count', types.IntegerType(), True)
])

In [ ]:
rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calc_revenue) \
    .map(unwrap) \
    .toDF()

In [ ]:
columns = ['VendorID', 'lpep_pickup_datetime','PULocationID','DOLocationID']

duration_rdd = df_green \
    .select(columns) \
    .rdd

In [ ]:
import pandas as pd

In [ ]:
rows = duration_rdd.take(10)

In [ ]:
def apply_model_in_batch(rows):
    df = pd.DataFrame(rows, columns=columns)
    cnt = len(df)
    return [cnt]
    # cnt = 0
    # for row in partition:
    #     cnt = cnt + 1
    # return cnt

In [ ]:
duration_rdd.mapPartitions(apply_model_in_batch).collect()